In [1]:
import pandas as pd

df_nodes = pd.read_csv('intermediate_nodes.csv')

In [2]:
df_weights = pd.read_csv('betting_team_probabilities_with_interest.csv')

In [3]:
df_timezones = pd.read_excel('../data_fifa.xlsx', sheet_name='country_TZ')

In [4]:
df_node_weights = df_nodes.merge(df_weights, how="cross")

In [5]:

name_replacements = {
    "South Korea": "Korea Republic",
    "Turkey": "Türkiye",
    "Czech Republic": "Czechia",
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
}
df_node_weights[["country", "opponent"]] = df_node_weights[["country", "opponent"]].replace(name_replacements)

In [6]:
for i,r in df_node_weights.iterrows():
    if len(df_timezones[df_timezones['Country'] == r['country']]) == 0:
        print(f"Timezone not found for {r['country']}")
    utc_offset = df_timezones[df_timezones['Country'] == r['country']]['UTC'].values[0]
    df_node_weights.at[i, 'broadcast_local_time_team1'] = (df_node_weights.at[i, 'hour_utc'] + utc_offset)%24
    if df_node_weights.at[i, 'broadcast_local_time_team1'] >= 8:
        df_node_weights.at[i, 'game_weight_for_country'] = 0

In [7]:
df_node_weights = df_node_weights.rename(columns={
    "country": "team1",
    "opponent": "team2",
})

df_node_weights = df_node_weights.drop(columns=[
    "p_win",
    "closeness",
    "trends_interest_per_capita",
    "broadcast_local_time_team1",
    "venue"
])

In [8]:
df_node_weights.to_csv('node_weights.csv', index=False)


In [41]:
games = df_node_weights[['team1', 'team2']].drop_duplicates()

In [42]:
print(games)

            team1           team2
0             USA         Türkiye
1             USA        Paraguay
2             USA       Australia
3    South Africa  Korea Republic
4    South Africa         Czechia
..            ...             ...
139       Curaçao         Ecuador
140       Curaçao         Germany
141    Cape Verde    Saudi Arabia
142    Cape Verde         Uruguay
143    Cape Verde           Spain

[144 rows x 2 columns]


In [53]:
df_simple = pd.DataFrame(columns=df_node_weights.columns)

In [ ]:
df_simple.to_csv('node_weights_simple.csv', index=False)

In [58]:
df = df_node_weights.copy()

df[["team1", "team2"]] = pd.DataFrame(
    sorted(pair) for pair in df[["team1", "team2"]].to_numpy()
)

group_cols = [
    col for col in df.columns
    if col != "game_weight_for_country"
]

df_node_weights_simple = (
    df
    .groupby(group_cols, as_index=False)
    .agg(game_weight_for_country=("game_weight_for_country", "sum"))
)

df_node_weights_simple.to_csv("node_weights_simple.csv", index=False)

df_node_weights_simple.head()

,date,stadium,hour_utc,team1,team2,commence_time,game_weight_for_country
0,2026-06-11,AT&T Stadium,0,Algeria,Argentina,2026-06-17T01:00:00Z,1683.277119
1,2026-06-11,AT&T Stadium,0,Algeria,Austria,2026-06-28T02:00:00Z,2585.364773
2,2026-06-11,AT&T Stadium,0,Algeria,Jordan,2026-06-23T03:00:00Z,2569.114785
3,2026-06-11,AT&T Stadium,0,Argentina,Austria,2026-06-22T17:00:00Z,398.667381
4,2026-06-11,AT&T Stadium,0,Argentina,Jordan,2026-06-28T02:00:00Z,401.414731
